In [ ]:
from omegaconf import OmegaConf
import torch
from PIL import Image
import numpy as np

from pathlib import Path

from diffusers.utils import make_image_grid
from model import DMD

import torch.nn.functional as F

from utils.distributed import (
    EMA_FSDP,
    fsdp_state_dict,
    fsdp_wrap,
    launch_distributed_job,
)

import os
from utils.dataset import ImageEditDataset, cycle
from utils.qwen_image_edit_wrapper import FAKE_LORA_NAME, GENERATOR_LORA_NAME


In [ ]:
CONFIG_PATH = Path("/home/rick-mbp/Diffusion-Distillation/configs/qwen_dmd.yaml")
DATASET_PATH = Path(
    "/home/rick-mbp/Diffusion-Distillation/qwen_training_data/qwen_image_edit_dataset_hard/metadata_edit.csv"
)
LOG_DIR = "./log"
WANDB = "./wandb"

In [ ]:
config = OmegaConf.load(CONFIG_PATH)


config.no_save = True
config.no_visualize = True

# get the filename of config_path


config.config_name = CONFIG_PATH.stem
config.logdir = LOG_DIR
config.wandb_save_dir = WANDB
config.disable_wandb = True


In [ ]:
dataset = ImageEditDataset(DATASET_PATH)

dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, num_workers=8)
dataloader = cycle(dataloader)


In [ ]:
model = DMD(config, "cuda")


### Debug Generator Gradient

In [ ]:
B, C, latent_H, latent_W = 1, 3, 128, 128

batch = next(dataloader)

img = batch["img"]
prompt = batch["prompts"]

# Latent shape is [B, 1, 16, H, W] (B, F, C, H, W)
latent_shape = [B, 1, 16, latent_H, latent_W]

img = batch["img"].to(model.device)  # Shape: [B, 3, 1024, 1024])
conditional_dict = model.encode_prompt(image=img, prompt=prompt)
unconditional_dict = model.encode_prompt(image=img, prompt="bad quality")

image_latent = model.run_vae_encoder(img)


# model.switch_to_fake()
# target_critic_params = [
#     param
#     for name, param in model.fake_score.named_parameters()
#     if param.requires_grad and FAKE_LORA_NAME in name
# ]

# critic_optimizer = torch.optim.AdamW(
#     target_critic_params,
#     lr=config.lr_critic if hasattr(config, "lr_critic") else config.lr,
#     betas=(config.beta1_critic, config.beta2_critic),
#     weight_decay=config.weight_decay,
# )

# model.switch_to_generator()

# target_generator_params = [
#     param
#     for name, param in model.generator.named_parameters()
#     if param.requires_grad and GENERATOR_LORA_NAME in name
# ]

# generator_optimizer = torch.optim.AdamW(
#     target_generator_params,
#     lr=config.lr,
#     betas=(config.beta1, config.beta2),
#     weight_decay=config.weight_decay,
# )


In [ ]:
pred_image, gradient_mask, denoised_timestep_from, denoised_timestep_to = (
    model._run_generator(
        image_or_video_shape=latent_shape,  # Pass latent shape
        conditional_dict=conditional_dict,
        initial_latent=image_latent,
    )
)


# # Step 2: Compute the DMD loss
dmd_loss, dmd_log_dict = model.compute_distribution_matching_loss(
    image_or_video=pred_image,
    conditional_dict=conditional_dict,
    unconditional_dict=unconditional_dict,
    gradient_mask=gradient_mask,
    denoised_timestep_from=denoised_timestep_from,
    denoised_timestep_to=denoised_timestep_to,
)


In [ ]:
model.zero_grad()
generator_loss, generator_log_dict = model.generator_loss(
    image_or_video_shape=[1, 3, 128, 128],
    conditional_dict=conditional_dict,
    unconditional_dict=unconditional_dict,
    initial_latent=image_latent,
)


In [ ]:
generator_loss.backward()

In [ ]:
params = []

for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"Layer: {name}")
    if param.grad is not None:
        params.append(param)
        print(f"Layer: {name} | Grad Mean: {param.grad.mean().item():.6f}")


In [ ]:
critic_loss, critic_log_dict = model.critic_loss(
    image_or_video_shape=[1, 3, 128, 128],
    conditional_dict=conditional_dict,
    unconditional_dict=unconditional_dict,
    initial_latent=image_latent,
)


In [ ]:
critic_loss.backward()


In [ ]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"Layer: {name}")
    if param.grad is not None:
        print(f"Layer: {name} | Grad Mean: {param.grad.mean().item():.7f}")


In [ ]:
# loss = F.mse_loss(image_latent, pred_image)
# loss.backward()

# dmd_loss.backward()

# critic_loss.backward()

generator_loss.backward()

In [ ]:
trainable_params = [p for p in model.parameters() if p.requires_grad]
grads = [p.grad for p in trainable_params if p.grad is not None]


In [ ]:
len(trainable_params), len(grads)


In [ ]:
trainable_params = [p for p in model.parameters() if p.requires_grad]
len(trainable_params)

In [ ]:
# generator_optimizer.zero_grad(set_to_none=True)

# List names of parameters that actually have a gradient
layers_with_grad = [
    name for name, param in model.named_parameters() if param.grad is not None
]

for name in layers_with_grad:
    print(f"Gradient found in: {name}")

# Total count for verification
print(f"\nTotal layers with gradients: {len(layers_with_grad)}")


In [ ]:
trainable_params = [p for p in model.parameters() if p.requires_grad]
grads = [p.grad for p in trainable_params if p.grad is not None]
grads

In [ ]:
device = "cuda"

image_latent = torch.randn(1, 16, 128, 128).to(torch.bfloat16).to(device)
batch = next(dataloader)

img = batch["img"]
prompt = batch["prompts"]
conditional_dict = model.encode_prompt(image=img, prompt=prompt)
unconditional_dict = model.encode_prompt(image=img, prompt="bad quality")
timestep = torch.Tensor([1]).to(device)

In [ ]:
generator_loss, generator_log_dict = model.generator_loss(
    image_or_video_shape=[1, 3, 128, 128],
    conditional_dict=conditional_dict,
    unconditional_dict=unconditional_dict,
    initial_latent=image_latent,
)


In [ ]:
critic_loss, critic_log_dict = model.critic_loss(
    image_or_video_shape=[1, 3, 128, 128],
    conditional_dict=conditional_dict,
    unconditional_dict=unconditional_dict,
    initial_latent=image_latent,
)

In [ ]:
model.qwen_image_edit_wrapper.set_adapter_trainable("fake", freeze_others=False)
model.qwen_image_edit_wrapper.set_adapter_trainable("real", freeze_others=False)


In [ ]:
# model.qwen_image_edit_wrapper.freeze_all()

model.qwen_image_edit_wrapper.switch_to_fake()

In [ ]:
for name, param in model.qwen_image_edit_wrapper.named_parameters():
    if "lora" in name:
        print(f"{name}: {param.requires_grad}")
        break

    print(f"{name}: {param.requires_grad}")


In [ ]:
from safetensors.torch import load_file, save_file


distill_tensors = load_file(
    "/home/rick-mbp/Diffusion-Distillation/qwen_distillation_data/dmd_test_2/checkpoint_model_000005/generator_lora.safetensors"
)
normal_tensors = load_file(
    "/mnt/model_training_disk/qwen_model_weight/ckpt/Qwen-Image-Edit-2509_lora-rank-32_lr-1e-4_hard/step-6000-hf.safetensors"
)

In [ ]:
for name, value in distill_tensors.items():
    normal_tensor = normal_tensors[name]

    if not normal_tensor.equal(value):
        print(name)


### Test EMA

In [ ]:
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
import torch


class EMA_FSDP:
    def __init__(self, fsdp_module: torch.nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = {}
        self._init_shadow(fsdp_module)

    @torch.no_grad()
    def _init_shadow(self, fsdp_module):
        # Handle FSDP case
        if isinstance(fsdp_module, FSDP):
            with FSDP.summon_full_params(
                fsdp_module, writeback=False, offload_to_cpu=True, rank0_only=True
            ):
                for n, p in fsdp_module.module.named_parameters():
                    self.shadow[n] = p.detach().clone().float().cpu()
        # Handle Standard/Single-GPU case
        else:
            for n, p in fsdp_module.named_parameters():
                if "generator" not in n:
                    continue
                self.shadow[n] = p.detach().clone().float().cpu()

    @torch.no_grad()
    def update(self, fsdp_module):
        d = self.decay

        # Helper to update shadow params
        def update_params(module_params):
            for n, p in module_params:
                if n in self.shadow:
                    self.shadow[n].mul_(d).add_(p.detach().float().cpu(), alpha=1.0 - d)

        if isinstance(fsdp_module, FSDP):
            with FSDP.summon_full_params(
                fsdp_module, writeback=False, offload_to_cpu=True, rank0_only=True
            ):
                update_params(fsdp_module.module.named_parameters())
        else:
            update_params(fsdp_module.named_parameters())

    def state_dict(self):
        return self.shadow

    def load_state_dict(self, sd):
        self.shadow = {k: v.clone() for k, v in sd.items()}

    def copy_to(self, fsdp_module):
        # load EMA weights into the generator
        if isinstance(fsdp_module, FSDP):
            with FSDP.summon_full_params(fsdp_module, writeback=True):
                for n, p in fsdp_module.module.named_parameters():
                    if n in self.shadow:
                        p.data.copy_(self.shadow[n].to(p.dtype, device=p.device))
        else:
            for n, p in fsdp_module.named_parameters():
                if n in self.shadow:
                    p.data.copy_(self.shadow[n].to(p.dtype, device=p.device))


In [ ]:
ema_weight = 0.99

generator_ema = EMA_FSDP(model.generator, decay=ema_weight)


In [ ]:
generator_ema.shadow.keys()